In [7]:
import pandas as pd, numpy as np, re
from pathlib import Path

BASE = Path("/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data")
PAST = BASE / "Combined Past Holdings"
OUT = BASE / "all_holdings_2017_2025.csv"
NET_ASSETS_XLSX = BASE / "All Net Assets .xlsx"

VALID_ETFS = {
    "ESGU","ESGD","ESGE","DSI","SUSA","USCL","PABU","ESML","ICLN","LCTU",
    "USXF","CRBN","SUSL","DMXF","XVV","XJH","LCTD","PABD","SDG","EMXF"
}
TICKER_MAP = {"MPCT":"SDG"}

def to_float(s):
    if s is None:
        return np.nan
    t = str(s).strip()
    if t in {"", "nan", "NA", "N/A", "-", "--", "—"}:
        return np.nan
    t = t.replace("(", "-").replace(")", "")
    t = re.sub(r"[^0-9.\-]", "", t)
    if t.count(".") > 1:
        first, *rest = t.split(".")
        t = first + "." + "".join(rest)
    try:
        return float(t)
    except:
        return np.nan

na = pd.read_excel(NET_ASSETS_XLSX)
na.columns = [c.strip().lower() for c in na.columns]
na = na.rename(columns={"etf ticker":"etf_ticker","net assets":"net_assets"})
req = {"etf_ticker","year","net_assets"}
missing = req - set(na.columns)
if missing:
    raise ValueError(f"Net Assets file missing columns: {missing}")
na["etf_ticker"] = na["etf_ticker"].astype(str).str.strip().str.upper().replace(TICKER_MAP)
na["year"] = na["year"].astype(int)
na["net_assets"] = na["net_assets"].map(to_float)
na = na[na["etf_ticker"].isin(VALID_ETFS) & na["net_assets"].notna()]

def get_year_from_name(p):
    m = re.search(r"soi_(\d{4})_final\.csv$", p.name, flags=re.I)
    if not m:
        raise ValueError(f"Year parse failed for {p.name}")
    return int(m.group(1))

def compute_weight_from_value(df, year):
    df = df.copy()
    df["etf_ticker"] = df["etf_ticker"].astype(str).str.upper().str.strip().replace(TICKER_MAP)
    df = df[df["etf_ticker"].isin(VALID_ETFS) & df["etf_ticker"].ne("") & df["etf_ticker"].ne("NAN")]
    df["__year__"] = int(year)
    df = df.merge(na, left_on=["etf_ticker","__year__"], right_on=["etf_ticker","year"], how="left")
    miss = df.loc[df["net_assets"].isna(), "etf_ticker"].dropna().unique().tolist()
    if miss:
        raise ValueError(f"Missing Net Assets for ETF(s) {miss} in year {year}")
    df["weight(%)"] = 100.0 * df["value"] / df["net_assets"]
    return df

def load_soi(path):
    year = get_year_from_name(path)
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    req = {"etf_ticker","etf_name","company_ticker","name_normalized","value"}
    if not req.issubset(df.columns):
        raise ValueError(f"Missing columns in {path.name}: {req - set(df.columns)}")
    df["value"] = df["value"].map(to_float).fillna(0.0)
    df = compute_weight_from_value(df, year)
    df = df[["etf_ticker","name_normalized","company_ticker","weight(%)"]].copy()
    df = df.groupby(["etf_ticker","name_normalized","company_ticker"], dropna=False, as_index=False)["weight(%)"].sum()
    df["date"] = f"{year}-12-31"
    df.rename(columns={"etf_ticker":"ETF_TICKER","name_normalized":"Normalised name","company_ticker":"Company_ticker"}, inplace=True)
    return df[["ETF_TICKER","date","Normalised name","Company_ticker","weight(%)"]]

def load_2025(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    wcol = None
    for c in ["weight (%)","weight(%)","weight_percent","weight","weight %","weight pct"]:
        if c in df.columns:
            wcol = c
            break
    if wcol is not None:
        df["weight(%)"] = df[wcol].apply(lambda x: to_float(str(x).replace("%","")))
        df["etf_ticker"] = df.get("etf_ticker", df.get("etf ticker")).astype(str).str.upper().str.strip().replace(TICKER_MAP)
    else:
        vcol = None
        for c in ["market value","market_value","value","value_usd","market_value_usd","market value (usd)"]:
            if c in df.columns:
                vcol = c
                break
        if vcol is None:
            raise ValueError("2025 file has neither a weight column nor a usable value column")
        df["value"] = df[vcol].map(to_float).fillna(0.0)
        df["etf_ticker"] = df.get("etf_ticker", df.get("etf ticker")).astype(str).str.upper().str.strip().replace(TICKER_MAP)
        df = compute_weight_from_value(df, 2025)
    if "name_normalized" not in df.columns and "name_normalised" in df.columns:
        df = df.rename(columns={"name_normalised":"name_normalized"})
    if "company_ticker" not in df.columns and "ticker" in df.columns:
        df = df.rename(columns={"ticker":"company_ticker"})
    df = df[df["etf_ticker"].isin(VALID_ETFS)]
    df = df[["etf_ticker","name_normalized","company_ticker","weight(%)"]].copy()
    df = df.groupby(["etf_ticker","name_normalized","company_ticker"], dropna=False, as_index=False)["weight(%)"].sum()
    df["date"] = "2025-12-31"
    df.rename(columns={"etf_ticker":"ETF_TICKER","name_normalized":"Normalised name","company_ticker":"Company_ticker"}, inplace=True)
    return df[["ETF_TICKER","date","Normalised name","Company_ticker","weight(%)"]]

past_files = sorted([p for p in PAST.glob("soi_20*_final.csv") if p.is_file()])
past_frames = [load_soi(p) for p in past_files]
cur_2025 = load_2025(BASE / "holdings_2025_final.csv")
all_holdings = pd.concat(past_frames + [cur_2025], ignore_index=True)
all_holdings["weight(%)"] = all_holdings["weight(%)"].astype(float).round(6)
all_holdings = all_holdings.sort_values(["ETF_TICKER","date","weight(%)"], ascending=[True,True,False]).reset_index(drop=True)
all_holdings.to_csv(OUT, index=False)
print(f"Wrote {OUT}")


Wrote /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data/all_holdings_2017_2025.csv
